# Round41 Final Extensive

Notebook para correr uma ronda extensiva durante a noite com os seis targets.

Foco por target:

- `1159_25`: garnish amarelo, prato de madeira, menos confusao nas rodelas.
- `1159_29`: ondas maiores, espuma e surf no primeiro plano.
- `1159_3`: armadura mais clara/anime e blade amarela curva.
- `1159_7`: grelha 4x4 de cubo mais visivel.
- `7836`: faixa/planeta diagonal rosado em vez de apenas galaxia.
- `9338`: conter teal na barriga e recuperar escalas arco-iris.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "search_multiseed_validate.py").exists():
    PROJECT_ROOT = Path("C:\\Users\\tugap\\Desktop\\Universidade\\Masters2\u00baAno\\IAG\\ProjetoCunha\\Projeto2")

SRC_DIR = PROJECT_ROOT / "src"
PROMPT_BANK = PROJECT_ROOT / "prompts" / "refinement_round41_final_extensive.json"
TARGETS_DIR = PROJECT_ROOT / "TP2-students" / "students" / "tp2-chosen"
OUTPUT_DIR = PROJECT_ROOT / "TP2-students" / "students" / "outputs"

PYTHON_CANDIDATES = [
    PROJECT_ROOT / ".venv_win" / "Scripts" / "python.exe",
    Path("C:\\Users\\tugap\\Desktop\\Universidade\\Masters2\u00baAno\\IAG\\Projeto 2\\.venv\\Scripts\\python.exe"),
    Path(sys.executable),
]

def has_module(python_exe, module_name):
    if not Path(python_exe).exists():
        return False
    result = subprocess.run(
        [str(python_exe), "-c", f"import {module_name}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    return result.returncode == 0

PYTHON_EXE = None
for candidate in PYTHON_CANDIDATES:
    if has_module(candidate, "diffusers"):
        PYTHON_EXE = candidate
        break

if PYTHON_EXE is None:
    raise RuntimeError("No Python with diffusers found. Create .venv_win and install requirements.txt")

print("Project root:", PROJECT_ROOT)
print("Notebook kernel:", sys.executable)
print("Render/search Python:", PYTHON_EXE)
print("Prompt bank:", PROMPT_BANK)
assert (SRC_DIR / "generate_round41_final_extensive.py").exists()
assert (SRC_DIR / "search_multiseed_validate.py").exists()
assert TARGETS_DIR.exists()

Project root: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2
Notebook kernel: c:\Users\tugap\AppData\Local\Programs\Python\Python311\python.exe
Render/search Python: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe
Prompt bank: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round41_final_extensive.json


## Configuracao

In [2]:
RUN_MODE = "overnight"  # smoke | balanced | overnight | deep

MODES = {
    "smoke": {
        "identity": "round41_smoke",
        "limit_per_target": 8,
        "top_k": 3,
        "stage1_save_k": 5,
        "validation_top_n": 4,
        "ensemble_per_metric": 1,
        "seed_offsets": [1],
    },
    "balanced": {
        "identity": "round41_balanced",
        "limit_per_target": 250,
        "top_k": 12,
        "stage1_save_k": 30,
        "validation_top_n": 20,
        "ensemble_per_metric": 5,
        "seed_offsets": [1, 2, 3],
    },
    "overnight": {
        "identity": "round41_final_extensive",
        "limit_per_target": None,
        "top_k": 16,
        "stage1_save_k": 60,
        "validation_top_n": 42,
        "ensemble_per_metric": 10,
        "seed_offsets": [1, 2, 3, 4, 5],
    },
    "deep": {
        "identity": "round41_final_deep",
        "limit_per_target": None,
        "top_k": 20,
        "stage1_save_k": 80,
        "validation_top_n": 60,
        "ensemble_per_metric": 12,
        "seed_offsets": [1, 2, 3, 4, 5, 6, 7],
    },
}

config = MODES[RUN_MODE]
print("Selected mode:", RUN_MODE)
print(json.dumps(config, indent=2))

Selected mode: overnight
{
  "identity": "round41_final_extensive",
  "limit_per_target": null,
  "top_k": 16,
  "stage1_save_k": 60,
  "validation_top_n": 42,
  "ensemble_per_metric": 10,
  "seed_offsets": [
    1,
    2,
    3,
    4,
    5
  ]
}


## Gerar prompts

In [3]:
cmd = [str(PYTHON_EXE), str(SRC_DIR / "generate_round41_final_extensive.py"), "--output", str(PROMPT_BANK)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

data = json.loads(PROMPT_BANK.read_text(encoding="utf-8"))
print({target: len(entries) for target, entries in data.items()})
print("Total prompts:", sum(len(entries) for entries in data.values()))

Running: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\generate_round41_final_extensive.py --output c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round41_final_extensive.json
{'1159_25.png': 500, '1159_29.png': 900, '1159_3.png': 750, '1159_7.png': 850, '7836.png': 900, '9338.png': 1000}
Total prompts: 4900


## Correr pesquisa

In [4]:
args = [
    str(PYTHON_EXE),
    str(SRC_DIR / "search_multiseed_validate.py"),
    "--prompts", str(PROMPT_BANK),
    "--targets", str(TARGETS_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--identity", config["identity"],
    "--top-k", str(config["top_k"]),
    "--stage1-save-k", str(config["stage1_save_k"]),
    "--validation-top-n", str(config["validation_top_n"]),
    "--ensemble-per-metric", str(config["ensemble_per_metric"]),
    "--seed-offsets", *[str(seed) for seed in config["seed_offsets"]],
    "--offline",
    "--disable-progress-bar",
    "--maxstack-scoring",
]
if config["limit_per_target"] is not None:
    args.extend(["--limit-per-target", str(config["limit_per_target"])])

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

print("Running:")
print(" ".join(args))
process = subprocess.Popen(
    args,
    cwd=PROJECT_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Search failed with exit code {return_code}")
print("Finished successfully")

Running:
C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\search_multiseed_validate.py --prompts c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round41_final_extensive.json --targets c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\tp2-chosen --output-dir c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs --identity round41_final_extensive --top-k 16 --stage1-save-k 60 --validation-top-n 42 --ensemble-per-metric 10 --seed-offsets 1 2 3 4 5 --offline --disable-progress-bar --maxstack-scoring
Couldn't connect to the Hub: Cannot reach https://huggingface.co/api/models/SimianLuo/LCM_Dreamshaper_v7: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable..
Will try to load from local cache.

## Ver resultados

In [ ]:
run_dirs = sorted(
    [path for path in OUTPUT_DIR.glob(f"*_{config['identity']}") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not run_dirs:
    print("No run directory found")
else:
    latest = run_dirs[0]
    print("Latest run:", latest)
    for item in sorted(latest.glob("*")):
        print(item.name)

Latest run: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs\20260603-021040_round41_final_extensive
1159_25
1159_25_robust_prompt_ranking.csv
1159_25_selected_for_multiseed.csv
1159_25_stage1_fixed_seed_top60.csv
1159_29
1159_29_robust_prompt_ranking.csv
1159_29_selected_for_multiseed.csv
1159_29_stage1_fixed_seed_top60.csv
1159_3
1159_3_robust_prompt_ranking.csv
1159_3_selected_for_multiseed.csv
1159_3_stage1_fixed_seed_top60.csv
1159_7
1159_7_robust_prompt_ranking.csv
1159_7_selected_for_multiseed.csv
1159_7_stage1_fixed_seed_top60.csv
7836
7836_robust_prompt_ranking.csv
7836_selected_for_multiseed.csv
7836_stage1_fixed_seed_top60.csv
9338
9338_robust_prompt_ranking.csv
9338_selected_for_multiseed.csv
9338_stage1_fixed_seed_top60.csv
contact_sheet_top16_robust.jpg
stage1_fixed_seed_metrics.csv
stage2_aux_seed_metrics.csv
summary.json
top16_robust_fixed_seed.csv


: 